In [7]:
import sys
import os
sys.path.insert(0,os.path.abspath('..'))

In [8]:
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request
from starlette.responses import Response,JSONResponse
from services.redisclient import rate_limit_check
from config.settings import settings

In [9]:
async def rateLimitMiddleware(app):
    async def middleware(scope,receive,send):
        if scope["type"]!="http":
            await app(scope,receive,send)
            return
        request=Request(scope,receive)
        if request.url.path.startswith("/api/v1/health"):
            await app(scope,receive,send)
            return
        userId=request.headers.get("X-User-Id",request.client.host or "anon")
        allowed=await rate_limit_check(userId,settings.RATELIMIT_INTERVAL)
        if not allowed:
            response=JSONResponse(
                status_code=429,
                content={"detail":"rate limit exceeded"},
            )
            await response(scope,receive,send)
            return
        await app(scope,receive,send)
    return middleware